In [2]:
import pandas as pd
import random

random.seed(42)

# Define data parameters
regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Books']
salespersons = ['Alice', 'Bob', 'Carol', 'David', 'Emma', 'Frank']

# Generate 200 sales transactions
data = {
    'transaction_id': range(1001, 1201),
    'region': [random.choice(regions) for _ in range(200)],
    'category': [random.choice(categories) for _ in range(200)],
    'salesperson': [random.choice(salespersons) for _ in range(200)],
    'sales_amount': [round(random.uniform(50, 5000), 2) for _ in range(200)],
    'customer_id': [random.randint(5000, 5100) for _ in range(200)]
}

df = pd.DataFrame(data)
print(df.head(10))
print(f"\nDataset shape: {df.shape}")


   transaction_id region     category salesperson  sales_amount  customer_id
0            1001  North  Electronics        Emma       4000.80         5075
1            1002  North        Books       Alice       3634.70         5084
2            1003   East  Electronics       David       4210.50         5025
3            1004  South  Electronics        Emma       4601.73         5054
4            1005  South     Clothing         Bob       4904.57         5014
5            1006  South     Clothing       Carol       2693.91         5069
6            1007  North       Sports       Alice       4539.30         5028
7            1008  North       Sports       Frank       2979.87         5082
8            1009   West       Sports       David       3331.85         5019
9            1010  North     Clothing       Alice        465.54         5034

Dataset shape: (200, 6)


Task 1: Basic Grouping and Single Aggregations

In [3]:
sales_amount_region = df.groupby('region')['sales_amount'].sum().reset_index()
print(sales_amount_region)
print("-------------------------------------")

transaction_count = df.groupby('category')['transaction_id'].count().reset_index()
print(transaction_count)
print("-------------------------------------")

avg_sales_amnt = df.groupby('salesperson')['sales_amount'].mean().reset_index()
print(avg_sales_amnt)
print("-------------------------------------")

sor_regional = sales_amount_region.sort_values('sales_amount',ascending = False).idxmax()
print(sor_regional)
print("-------------------------------------")


  region  sales_amount
0   East      95189.81
1  North     135181.16
2  South     158977.36
3   West     109383.07
-------------------------------------
        category  transaction_id
0          Books              39
1       Clothing              42
2    Electronics              45
3  Home & Garden              43
4         Sports              31
-------------------------------------
  salesperson  sales_amount
0       Alice   1998.432273
1         Bob   2554.919063
2       Carol   2454.368571
3       David   2743.036444
4        Emma   2493.457273
5       Frank   2464.135750
-------------------------------------
region          3
sales_amount    2
dtype: int64
-------------------------------------


Task 2: Multi-Column Grouping and Multiple Aggregations

In [4]:
#Group by both region AND category to calculate total sales for each combination using

reg_cate = df.groupby(['region','category'])['sales_amount'].sum().reset_index
print(reg_cate)

<bound method Series.reset_index of region  category     
East    Books            20027.53
        Clothing         19926.20
        Electronics      22791.55
        Home & Garden    15949.08
        Sports           16495.45
North   Books            42592.53
        Clothing         18959.06
        Electronics      38889.84
        Home & Garden    19344.48
        Sports           15395.25
South   Books            21007.68
        Clothing         50878.36
        Electronics      39229.63
        Home & Garden    22592.00
        Sports           25269.69
West    Books            20359.41
        Clothing         16548.81
        Electronics      13553.27
        Home & Garden    33069.36
        Sports           25852.22
Name: sales_amount, dtype: float64>


In [9]:
df.groupby('salesperson')['sales_amount'].agg(['sum', 'mean', 'count']).reset_index()

,salesperson,sum,mean,count
3,David,123436.64,2743.036444,45
5,Frank,98565.43,2464.135750,40
4,Emma,82284.09,2493.457273,33
1,Bob,81757.41,2554.919063,32
2,Carol,68722.32,2454.368571,28
0,Alice,43965.51,1998.432273,22


In [13]:
#Sort the salesperson results by total sales in descending order to identify the top performer
df.groupby('salesperson')['sales_amount'].agg(['sum', 'mean', 'count']).reset_index().sort_values(by='sum',ascending=False)

,salesperson,sum,mean,count
3,David,123436.64,2743.036444,45
5,Frank,98565.43,2464.135750,40
4,Emma,82284.09,2493.457273,33
1,Bob,81757.41,2554.919063,32
2,Carol,68722.32,2454.368571,28
0,Alice,43965.51,1998.432273,22


In [22]:
#se .idxmax() on the grouped category sales to find which category has the maximum total revenue

df.groupby('category')['sales_amount'].sum().idxmax()

'Electronics'

Task 3: Custom Aggregation and Complete Sales Report

In [34]:
#Define a custom aggregation function that calculates the sales range (max - min) for each group:

def aggr_func(x):
    return x.max() - x.min()

# Apply custom function with groupby
res = df.groupby('salesperson')['sales_amount'].agg([aggr_func])
res1 = df.groupby('region')['sales_amount'].agg([aggr_func])
res2 = df.groupby('category')['sales_amount'].agg([aggr_func])

print("Groupby SalesPerson\n",res)
print("\nGroupby Region\n",res1)
print("\nGroupby category\n",res2)


Groupby SalesPerson
              aggr_func
salesperson           
Alice          4403.95
Bob            4791.94
Carol          4691.83
David          4545.19
Emma           4734.05
Frank          4650.24

Groupby Region
         aggr_func
region           
East      4450.64
North     4677.85
South     4609.88
West      4699.93

Groupby category
                aggr_func
category                
Books            4398.90
Clothing         4872.62
Electronics      4608.92
Home & Garden    4667.19
Sports           4790.90


In [35]:
#Apply this custom function along with standard aggregations to analyze sales by region

def sales_range(x):
  return x.max() - x.min()

res = df.groupby('region')['sales_amount'].agg(['sum', 'mean', 'min', 'max', sales_range]).reset_index()
print(res)

  region        sum         mean     min      max  sales_range
0   East   95189.81  2213.716512  307.82  4758.46      4450.64
1  North  135181.16  2413.949286  110.71  4788.56      4677.85
2  South  158977.36  2789.076491  373.45  4983.33      4609.88
3   West  109383.07  2485.978864  203.60  4903.53      4699.93


In [41]:
#Create a final summary report that shows for each region:

print("Final Summary Report For each Region=====\n")
df.groupby('region').agg({
    'sales_amount': ['sum', 'mean'],
    'customer_id': 'count'
}).reset_index()


Final Summary Report For each Region=====



region sales_amount              customer_id
                  sum         mean       count
0   East     95189.81  2213.716512          43
1  North    135181.16  2413.949286          56
2  South    158977.36  2789.076491          57
3   West    109383.07  2485.978864          44

In [42]:
#for understanding
print("Final Summary Report For each Region with Custom columnNAme=====\n")
df.groupby('region').agg(
    totalSales = ('sales_amount','sum'),
    AvgSales = ('sales_amount','mean'),
     CustomerCount = ('customer_id', 'count')
).reset_index()

Final Summary Report For each Region with Custom columnNAme=====



,region,totalSales,AvgSales,CustomerCount
0,East,95189.81,2213.716512,43
1,North,135181.16,2413.949286,56
2,South,158977.36,2789.076491,57
3,West,109383.07,2485.978864,44




dictionary-based aggregation produces multiple column structure because different mathematical / custom function performed over different columns.

Single Aggreation- leads to flat structure columns as it perform mathematical fucntion over the same column